In [9]:
import pandas as pd
import seaborn as sns
import numpy as np
import os
import tqdm

In [ ]:
# define some global objects
DEFAULT_USER_BASE_DIR = "C:/Users/hychu/OneDrive/Desktop/Summer25"
DEFAULT_FOLDSEEK_PATH = os.path.join(DEFAULT_USER_BASE_DIR, "homolog-results/foldseek")
DEFAULT_INPUT_FILE = os.path.join(DEFAULT_USER_BASE_DIR, "github/ppi-classifiers/data/training_features.csv")
DEFAULT_CACHE_FILE = os.path.join(DEFAULT_USER_BASE_DIR, "github/ppi-classifiers/data/homologs_tree.pkl")   
NUM_PROTEINS = 12343 # based on total count of tsv output files'

def parse_tsv_file(tsv_name, path) -> np.ndarray: 
    """ function to parse .tsv files that store Foldseek-identified hit (homolog) information for a given protein.
        - tsv_name: name of the .tsv file
        - path: home path for the .tsv file
        - ~params: list of params (by index) to consider from the results. DEFAULT = ['query', 'target', 'pident', 'fident']~
    """
    # verify that the file exists first and foremost
    FILE_PATH = os.path.join(path, tsv_name)
    if not os.path.exists(FILE_PATH):
        print(f"Error: cannot find file {tsv_name} in path {path}. Exiting.")
        return np.empty((0, 4))

    homologs_info = []
    try:
        with open(FILE_PATH, "r", newline = '', encoding = 'utf-8') as infile:
            lines = infile.read().splitlines()
            if not lines: # file is empty; no homologs detected by Foldseek
                # print(f"Empty .tsv file detected: {tsv_name} contains no homologs.")
                return np.empty((0, 4))
            
            n_homologs = len(lines) # for assertion
            for line in lines:
                # for each line, identify parameters of interest and append it to list of homologs
                items = line.strip().split('\t')
                query, target = items[0], items[1]
                try:
                    pident, fident = float(items[2]), float(items[3])
                    evalue, bits = float(items[13]), float(items[14])
                except ValueError:
                    continue
                homologs_info.append([query, target, pident, fident, evalue, bits])
            # convert from nested list structure to np.2darray
            homologs_info = np.array(homologs_info, dtype = object)
            assert len(homologs_info) == n_homologs, \
                f"Error: homolog counts do not match: {n_homologs} lines vs. {len(homologs_info)} parsed."
            # print(f"Homologs detected and parsed for file {tsv_name}.")
    except Exception as e:
        print(f"Error parsing {tsv_name}: {e}")
    
    return homologs_info

tsv_files = []
try:
    for dirpath, dirnames, filenames in os.walk(DEFAULT_FOLDSEEK_PATH):
        for filename in tqdm.tqdm(filenames):
            if filename.endswith('.tsv'):
                tsv_files.append(os.path.join(dirpath, filename))
    tsv_files = np.array(tsv_files, dtype = object)
except Exception as e:
    print(f"Error extracting .tsv file names: {e}")
assert len(tsv_files) == NUM_PROTEINS, \
    f"Error: mismatch in .tsv file count. Expected {NUM_PROTEINS}, got {len(tsv_files)}."



  0%|          | 0/12343 [00:00<?, ?it/s]

100%|██████████| 12343/12343 [01:24<00:00, 145.62it/s]


In [16]:
homologs_tree = {}
proteins = [tsv.replace('.tsv', '').strip().split('\\')[-1] for tsv in tsv_files]
for protein_id in tqdm.tqdm(proteins):
    homologs_tree[protein_id] = parse_tsv_file(f"{protein_id}.tsv", DEFAULT_FOLDSEEK_PATH)
assert len(homologs_tree) == NUM_PROTEINS, \
    f"Error: incomplete homolog compilation. Expected {NUM_PROTEINS}, got {len(homologs_tree)}."

100%|██████████| 12343/12343 [00:09<00:00, 1337.59it/s]


In [17]:
homologs_tree

{'A0A022MQ12': array([['A0A022MQ12', '6sj4-assembly1.cif.gz_A', 100.0, 1.0, 1.486e-73,
         3191.0],
        ['A0A022MQ12', '4v1y-assembly1.cif.gz_B', 20.5, 0.205, 3.842e-31,
         1393.0],
        ['A0A022MQ12', '4f0r-assembly1.cif.gz_A', 22.3, 0.223, 2.306e-30,
         1360.0],
        ...,
        ['A0A022MQ12', '3hbl-assembly1.cif.gz_B', 13.3, 0.133, 0.0005147,
         243.0],
        ['A0A022MQ12', '6su1-assembly1.cif.gz_A', 11.3, 0.113, 0.0009354,
         232.0],
        ['A0A022MQ12', '4hnv-assembly1.cif.gz_C', 10.9, 0.109, 0.0009354,
         232.0]], dtype=object),
 'A0A024FSR7': array([], shape=(0, 4), dtype=float64),
 'A0A024L8R9': array([['A0A024L8R9', '6yst-assembly1.cif.gz_y', 100.0, 1.0, 3.169e-21,
         809.0],
        ['A0A024L8R9', '6ysu-assembly1.cif.gz_y', 100.0, 1.0, 1.17e-20,
         789.0],
        ['A0A024L8R9', '7jss-assembly1.cif.gz_8', 93.8, 0.938, 1.073e-15,
         614.0],
        ['A0A024L8R9', '4v95-assembly1.cif.gz_AY', 91.4, 0.914, 1.694e